# LLM-based knowledge extraction from Marker-PDF markdown outputs.

In [19]:
# =========================
# Import libraries
# =========================
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from openai import OpenAI

In [20]:
# =========================
# Chunk schema + prompt
# =========================
NODE_TYPE = "Publication"

CHUNK_TYPES = [
    "Background",
    "Research Goal",
    "Problem",
    "Method",
    "Data",
    "Result",
    "Discussion-Limitation",
    "Conclusion",
]

# You can tune these caps later
MAX_ITEMS_PER_TYPE: Dict[str, int] = {
    "Background": 3,
    "Research Goal": 5,
    "Problem": 5,
    "Method": 8,
    "Data": 6,
    "Result": 10,
    "Discussion-Limitation": 6,
    "Conclusion": 3,
}

# Hard safety limits (also enforced in code)
MAX_CHARS_PER_CHUNK = 1500
MIN_CHARS_PER_CHUNK = 120
MAX_WORDS_SUPPORTING_QUOTE = 50

SYSTEM_MSG = (
    "You are a precise extractor of knowledge chunks from academic papers. "
    "Return ONLY valid JSON. No prose, no markdown, no extra keys."
)

EXTRACTION_INSTRUCTIONS = f"""\
Extract knowledge chunks from the paper content, grouped by the following chunk types:
{", ".join([f'"{t}"' for t in CHUNK_TYPES])}

Return EXACTLY this JSON schema (same keys), with these rules:
- If a field is missing/unclear, return an empty value:
  - strings: ""
  - lists: []
- Do NOT add any extra keys.
- Do NOT include explanations.
- Do NOT include "Abstract". (It is handled elsewhere.)

Schema:
{{
  "chunks": [
    {{
      "chunk_type": "",
      "text": "",
      "supporting_quote": ["", ""],
      "section_hint": ""
    }}
  ]
}}

Extraction rules:
1) Valid chunk_type MUST be one of: {CHUNK_TYPES}
2) You may output multiple chunks per chunk_type (0..N).
3) Each chunk MUST be grounded in the paper:
   - supporting_quote must be a list of short verbatim quotes from the paper (<= {MAX_WORDS_SUPPORTING_QUOTE} words each).
   - The list of supporting_quote MUST directly support the chunk text.
   - The supporting_quote elements must include the key term(s) or number(s) used in the chunk text.
   - If you cannot provide a valid supporting_quote found in the paper, DO NOT output that chunk.
4) text must be a clean, self-contained paraphrase of the idea (not a long paste), typically 3–6 sentences.
5) Avoid redundancy: each chunk should represent a distinct idea.
6) Respect caps (max chunks per type). If there are more candidates, keep the most central ones:
{json.dumps(MAX_ITEMS_PER_TYPE, indent=2)}
7) Length constraints:
   - text length <= {MAX_CHARS_PER_CHUNK} characters.
   - text length >= {MIN_CHARS_PER_CHUNK} characters (otherwise omit the chunk).
8) section_hint should be a short label like: "Introduction", "Methods", "Results", "Discussion", "Conclusion", or "".

Output ONLY the JSON object.
"""

In [21]:
# =========================
# Marker path helpers (same pattern as your metadata notebook)
# =========================
def marker_markdown_path(base_dir: Path, paper_id: str) -> Path:
    return base_dir / paper_id / "markdown" / f"{paper_id}_md.md"


def list_paper_ids(base_dir: Path) -> List[str]:
    out: List[str] = []
    for p in base_dir.iterdir():
        if p.is_dir():
            out.append(p.name)

    def key(x: str) -> Tuple[int, Any]:
        try:
            return (0, int(x))
        except Exception:
            return (1, x)

    return sorted(out, key=key)

In [22]:
# =========================
# Utilities: truncation + robust JSON parsing + cleaning
# =========================
def safe_json_default() -> Dict[str, Any]:
    return {"chunks": []}


def compress_md_text(md_text: str, *, head_chars: int = 80_000, tail_chars: int = 60_000) -> str:
    """
    Papers can be long. This keeps both beginning and end, which helps capture
    Methods/Results/Discussion/Conclusion that often appear later.
    Adjust head/tail if you want.
    """
    if len(md_text) <= head_chars + tail_chars + 200:
        return md_text

    head = md_text[:head_chars]
    tail = md_text[-tail_chars:]
    return (
        "PAPER_CONTENT_HEAD_START\n"
        f"{head}\n"
        "PAPER_CONTENT_HEAD_END\n\n"
        "...\n\n"
        "PAPER_CONTENT_TAIL_START\n"
        f"{tail}\n"
        "PAPER_CONTENT_TAIL_END\n"
    )


def _strip_code_fences(text: str) -> str:
    # If model accidentally wraps JSON in ```json ... ```
    text = text.strip()
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip()


def try_parse_json(text: str) -> Optional[Dict[str, Any]]:
    """
    Best-effort JSON parse. Tries:
    1) direct json.loads
    2) substring from first '{' to last '}'
    """
    if not text:
        return None

    cleaned = _strip_code_fences(text)

    # 1) direct
    try:
        obj = json.loads(cleaned)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    # 2) salvage substring
    try:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            obj = json.loads(cleaned[start : end + 1])
            return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    return None


def limit_words(s: str, max_words: int) -> str:
    if not isinstance(s, str):
        return ""
    words = s.strip().split()
    if len(words) <= max_words:
        return s.strip()
    return " ".join(words[:max_words]).strip()

def normalize_supporting_quotes(val: Any) -> List[str]:
    """
    Returns list[str] of cleaned non-empty quotes.
    Accepts: list | str | None | other
    """
    if val is None:
        return []
    if isinstance(val, str):
        s = val.strip()
        return [s] if s else []
    if isinstance(val, list):
        out = []
        for x in val:
            if isinstance(x, str):
                s = x.strip()
                if s:
                    out.append(s)
        return out
    return []

def normalize_chunks(
    paper_id: str,
    data: Dict[str, Any],
) -> List[Dict[str, Any]]:
    """
    Enforce:
    - schema keys
    - chunk_type enum
    - caps per type
    - length constraints
    - supporting_quote word cap
    """
    chunks_raw = data.get("chunks")
    if not isinstance(chunks_raw, list):
        return []

    # Bucket by type to enforce caps
    buckets: Dict[str, List[Dict[str, Any]]] = {t: [] for t in CHUNK_TYPES}

    for c in chunks_raw:
        if not isinstance(c, dict):
            continue

        chunk_type = (c.get("chunk_type") or "").strip()
        if chunk_type not in buckets:
            continue

        '''
        text = (c.get("text") or "").strip()
        quote = (c.get("supporting_quote") or "").strip()
        section_hint = (c.get("section_hint") or "").strip()

        # supporting_quote is mandatory for inclusion
        if not quote:
            continue

        # Apply constraints
        quote = limit_words(quote, MAX_WORDS_SUPPORTING_QUOTE)
        '''

        text = (c.get("text") or "").strip()
        quotes = normalize_supporting_quotes(c.get("supporting_quote"))
        section_hint = (c.get("section_hint") or "").strip()

        # supporting_quote is mandatory for inclusion
        if not quotes:
            continue

        # Apply constraints (cap words per quote)
        quotes = [limit_words(q, MAX_WORDS_SUPPORTING_QUOTE) for q in quotes]
        quotes = [q for q in quotes if q]  # por si algo queda vacío
        if not quotes:
            continue

        if len(text) > MAX_CHARS_PER_CHUNK:
            text = text[:MAX_CHARS_PER_CHUNK].rstrip()

        if len(text) < MIN_CHARS_PER_CHUNK:
            continue

        buckets[chunk_type].append(
            {
                "paper_id": paper_id,
                "node_type": NODE_TYPE,
                "chunk_type": chunk_type,
                "text": text,
                "supporting_quote": quotes,
                "section_hint": section_hint,
            }
        )

    # Enforce caps + generate chunk_id
    normalized: List[Dict[str, Any]] = []
    for t in CHUNK_TYPES:
        max_n = MAX_ITEMS_PER_TYPE.get(t, 0)
        items = buckets[t]
        if max_n > 0:
            items = items[:max_n]

        for i, item in enumerate(items, start=1):
            item["chunk_id"] = f"{paper_id}__{t.replace(' ', '_').replace('-', '_')}__{i}"
            normalized.append(item)

    return normalized


def create_response_with_fallback(
    client: OpenAI,
    *,
    model: str,
    input_items: List[Dict[str, str]],
    reasoning_effort: str = "low",
    temperature: float = 0.0,
    max_output_tokens: int = 2500,
) -> Any:
    """
    Some models may reject certain parameters. This wrapper retries with fewer knobs.
    """
    # Attempt 1: full parameters
    try:
        return client.responses.create(
            model=model,
            input=input_items,
            text={"format": {"type": "json_object"}},
            reasoning={"effort": reasoning_effort},
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )
    except Exception as e1:
        msg1 = str(e1)

    # Attempt 2: drop temperature
    try:
        return client.responses.create(
            model=model,
            input=input_items,
            text={"format": {"type": "json_object"}},
            reasoning={"effort": reasoning_effort},
            max_output_tokens=max_output_tokens,
        )
    except Exception as e2:
        msg2 = str(e2)

    # Attempt 3: drop reasoning too
    try:
        return client.responses.create(
            model=model,
            input=input_items,
            text={"format": {"type": "json_object"}},
            max_output_tokens=max_output_tokens,
        )
    except Exception as e3:
        # Raise a combined error (useful for your skipped log)
        raise RuntimeError(
            "All OpenAI call attempts failed.\n"
            f"[attempt1] {msg1}\n"
            f"[attempt2] {msg2}\n"
            f"[attempt3] {type(e3).__name__}: {e3}"
        )

In [23]:
# =========================
# Core extraction logic
# =========================
def extract_one_paper(
    client: OpenAI,
    model: str,
    paper_id: str,
    md_text: str,
    *,
    head_chars: int = 80_000,
    tail_chars: int = 60_000,
    reasoning_effort: str = "low",
    max_output_tokens: int = 2500,
) -> List[Dict[str, Any]]:
    """
    One-call extraction that returns a FLAT list of chunks for this paper_id.
    (No Abstract extraction.)
    """
    md_text_compact = compress_md_text(md_text, head_chars=head_chars, tail_chars=tail_chars)

    user_content = (
        "PAPER_CONTENT_START\n"
        f"{md_text_compact}\n"
        "PAPER_CONTENT_END\n\n"
        f"{EXTRACTION_INSTRUCTIONS}"
    )

    resp = create_response_with_fallback(
        client,
        model=model,
        input_items=[
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": user_content},
        ],
        reasoning_effort=reasoning_effort,
        temperature=0.0,
        max_output_tokens=max_output_tokens,
    )

    # Handle incomplete responses early (likely truncated JSON)
    if getattr(resp, "status", None) == "incomplete":
        details = getattr(resp, "incomplete_details", None)
        raise RuntimeError(f"incomplete_response: {details}")

    text = (getattr(resp, "output_text", "") or "").strip()
    if not text:
        return []

    data = try_parse_json(text)
    if data is None:
        raise ValueError("json_parse_failed")

    return normalize_chunks(paper_id=paper_id, data=data)

In [24]:
# =========================
# Pipeline (incremental, resumable)
# =========================
def run_pipeline(
    base_dir: Path,
    output_file: Path,
    model: str,
    *,
    sleep_s: float = 0.2,
    limit: int = 0,
    head_chars: int = 80_000,
    tail_chars: int = 60_000,
    reasoning_effort: str = "low",
    max_output_tokens: int = 2500,
) -> None:
    load_dotenv()
    client = OpenAI()

    # 1) Load previous progress or init
    if output_file.exists():
        try:
            payload = json.loads(output_file.read_text(encoding="utf-8"))
            print(f"📖 Loading existing progress from {output_file}")
        except json.JSONDecodeError:
            print(f"⚠️ {output_file} is corrupt. Starting from scratch.")
            payload = {
                "version": "paper_chunks_v1",
                "model": model,
                "base_dir": str(base_dir.resolve()),
                "count": 0,
                "skipped": [],
                "records": [],
            }
    else:
        payload = {
            "version": "paper_chunks_v1",
            "model": model,
            "base_dir": str(base_dir.resolve()),
            "count": 0,
            "skipped": [],
            "records": [],
        }

    # Already processed papers
    processed_ids = {rec["paper_id"] for rec in payload.get("records", []) if isinstance(rec, dict) and "paper_id" in rec}

    paper_ids = list_paper_ids(base_dir)
    if limit and limit > 0:
        paper_ids = paper_ids[:limit]

    for pid in paper_ids:
        if pid in processed_ids:
            continue

        md_path = marker_markdown_path(base_dir, pid)
        if not md_path.exists():
            payload["skipped"].append({"paper_id": pid, "reason": f"missing_markdown: {str(md_path)}"})
            output_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
            continue

        try:
            print(f"📄 Processing: {pid}...")
            md_text = md_path.read_text(encoding="utf-8", errors="replace")

            chunks = extract_one_paper(
                client=client,
                model=model,
                paper_id=pid,
                md_text=md_text,
                head_chars=head_chars,
                tail_chars=tail_chars,
                reasoning_effort=reasoning_effort,
                max_output_tokens=max_output_tokens,
            )

            record = {
                "paper_id": pid,
                "chunks": chunks,  # flat list with paper_id repeated inside each chunk too
                "chunk_count": len(chunks),
            }

            payload["records"].append(record)
            payload["count"] = len(payload["records"])

            output_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
            processed_ids.add(pid)

        except Exception as e:
            error_msg = f"error: {type(e).__name__}: {e}"
            payload["skipped"].append({"paper_id": pid, "reason": error_msg})
            output_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

        time.sleep(sleep_s)

    print(f"✅ Pipeline finished. Total papers in file: {payload['count']}")

In [26]:
# =========================
# Run
# =========================
run_pipeline(
    base_dir=Path("publications/output_marker/LLM_gpt-4o-mini"),
    output_file=Path("paper_chunks.json"),
    model="gpt-5.2",
    sleep_s=0.2,
    limit=0,
    head_chars=700_000,
    tail_chars=0,
    reasoning_effort="low",
    max_output_tokens=25000,
)


📄 Processing: 1...
📄 Processing: 5...
📄 Processing: 7...
📄 Processing: 9...
📄 Processing: 10...
📄 Processing: 11...
📄 Processing: 12...
📄 Processing: 15...
📄 Processing: 16...
📄 Processing: 17...
📄 Processing: 18...
📄 Processing: 19...
📄 Processing: 20...
📄 Processing: 21...
📄 Processing: 22...
📄 Processing: 23...
📄 Processing: 24...
📄 Processing: 27...
📄 Processing: 28...
📄 Processing: 29...
📄 Processing: 30...
📄 Processing: 33...
📄 Processing: 34...
📄 Processing: 36...
📄 Processing: 37...
📄 Processing: 38...
📄 Processing: 39...
📄 Processing: 43...
📄 Processing: 45...
📄 Processing: 46...
📄 Processing: 49...
📄 Processing: 50...
📄 Processing: 51...
📄 Processing: 52...
📄 Processing: 54...
📄 Processing: 56...
📄 Processing: 57...
📄 Processing: 58...
📄 Processing: 59...
📄 Processing: 60...
📄 Processing: 61...
📄 Processing: 62...
📄 Processing: 64...
📄 Processing: 65...
📄 Processing: 66...
📄 Processing: 67...
📄 Processing: 69...
📄 Processing: 70...
📄 Processing: 71...
📄 Processing: 72...
📄 Pr